## SDK 0.3.0 설치 / Install SDK 0.3.0
첫 셀은 SDK 0.3.0이 없을 때만 고정 wheel을 설치합니다(네트워크 필요). 설치된 경우 건너뜁니다. 결과 전송은 별도 선택이며 기본 False입니다.
The first cell installs the pinned wheel only if SDK 0.3.0 is absent (network required). If 0.3.0 is already installed, installation is skipped. Publishing results is a separate choice, default False.


In [ ]:
# Install only when this SDK version is absent. No result publication.
import importlib
import subprocess
import sys

try:
    import luna_repro
    current_sdk = getattr(luna_repro, "__version__", None)
except ModuleNotFoundError as error:
    if error.name != "luna_repro":
        raise
    current_sdk = None

if current_sdk != "0.3.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "https://github.com/lunalab-ai/luna-repro-sdk/releases/download/v0.3.0/luna_repro-0.3.0-py3-none-any.whl#sha256=4fc3b6a0ff1d78dbb2ace0ea08dcedc3794e14f3930d0c60f29c42fbb04e2489"])
    if current_sdk is not None:
        raise RuntimeError("SDK updated. Restart the runtime/kernel, then Run all again. SDK 업데이트 후 런타임을 재시작하고 다시 실행하세요.")
    importlib.invalidate_caches()
    import luna_repro
assert luna_repro.__version__ == "0.3.0"
print("LUNA Repro SDK", luna_repro.__version__)


# 실행 중 결과 쌓기 / Incremental collection
SDK를 설치한 뒤 실행하세요. 같은 page 객체에 선택한 결과만 쌓습니다. 기본 예제는 합성 CPU 계산과 로컬 저장만 수행합니다.
Install the SDK first. Add explicit results to the same page. This example performs synthetic CPU computation and local saves only.

In [ ]:
import os
import luna_repro as repro
page = repro.page(os.environ.get('LUNA_REPRO_PROJECT', '12345678-1234-5678-1234-567812345678'), '누적 합성 실험 / Incremental toy', evidence={'origin': 'synthetic', 'execution_status': 'unknown', 'evaluation_status': 'not_run'})
page.section('setup', '실험 조건 / Conditions', source={'file': 'incremental.ipynb', 'cell_id': 'prepare-page'}).text('Synthetic CPU example; no GPU, model or dataset download.')
page.save('.luna-repro/incremental-partial.json')

In [ ]:
reference = [0.0, 0.5, 1.0]
prediction = [0.1, 0.4, 0.9]
mae = sum(abs(a-b) for a,b in zip(reference, prediction)) / len(reference)
page.section('quality', '계산 결과 / Results', source={'file': 'incremental.ipynb', 'cell_id': 'compute'}).metrics({'MAE': mae}).table(['reference', 'prediction'], list(zip(reference, prediction)))
page.evidence.update(execution_status='completed', evaluation_status='completed')

In [ ]:
page.save('.luna-repro/incremental-result.json')
# 연결 설정 후 명시적 전송 / Explicitly publish after configuring the connection:
# receipt = page.publish()
# print(receipt.get('url') or receipt['message'])